In [1]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

OUTPUT_DIR = Path.cwd().parent / "output"
model_names = sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_dir())

model_dropdown = widgets.Dropdown(options=model_names, description="Model:")
display(model_dropdown)

Dropdown(description='Model:', options=('baseline', 'curriculum_baby_steps', 'curriculum_baby_steps_inverse', …

In [2]:
import json

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def get_best_checkpoint(model_dir: Path) -> Path:
    state_paths = [
        model_dir / "trainer_state.json",
        *model_dir.glob("checkpoint-*/trainer_state.json"),
    ]
    for state_path in state_paths:
        if not state_path.exists():
            continue
        with state_path.open() as state_file:
            best_checkpoint = json.load(state_file).get("best_model_checkpoint")
        if best_checkpoint:
            checkpoint_dir = Path(best_checkpoint)
            if checkpoint_dir.exists():
                return checkpoint_dir
            local_checkpoint_dir = model_dir / checkpoint_dir.name
            if local_checkpoint_dir.exists():
                return local_checkpoint_dir
    raise FileNotFoundError(f"No best checkpoint found in {model_dir}")


def predict(premise: str, hypothesis: str) -> tuple[int, list[float]]:
    checkpoint_dir = get_best_checkpoint(OUTPUT_DIR / model_dropdown.value)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    model.eval()

    inputs = tokenizer(premise, hypothesis, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    predicted_label = int(probs.argmax())
    return predicted_label, probs.tolist()

In [3]:
premise_input = widgets.Text(description="Premise:", placeholder="Enter premise")
hypothesis_input = widgets.Text(
    description="Hypothesis:", placeholder="Enter hypothesis"
)
run_button = widgets.Button(description="Test model")
output_area = widgets.Output()

label_names = {0: "NONE", 1: "ENTAILMENT"}
label_marks = {0: ("✗", "red"), 1: ("✓", "green")}


def on_run_clicked(_):
    output_area.clear_output()
    with output_area:
        premise = premise_input.value
        hypothesis = hypothesis_input.value
        predicted_label, probs = predict(premise, hypothesis)
        mark, color = label_marks[predicted_label]
        print(f"Premise: {premise}")
        print(f"Hypothesis: {hypothesis}")
        display(
            widgets.HTML(
                f'<span style="color: {color}; font-weight: bold;">{mark}</span> '
                f"Predicted label: {label_names[predicted_label]}"
            )
        )
        print(f"Probabilities: {probs}")


run_button.on_click(on_run_clicked, remove=True)
run_button.on_click(on_run_clicked)
display(premise_input, hypothesis_input, run_button, output_area)

Text(value='', description='Premise:', placeholder='Enter premise')

Text(value='', description='Hypothesis:', placeholder='Enter hypothesis')

Button(description='Test model', style=ButtonStyle())

Output()

In [6]:
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

checkpoint_dir = get_best_checkpoint(OUTPUT_DIR / model_dropdown.value)
print(f"Evaluating checkpoint: {checkpoint_dir.name}")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

test_dataset = load_dataset("nilc-nlp/assin2", split="test")
tokenized_test_dataset = test_dataset.map(
    lambda batch: tokenizer(
        batch["premise"], batch["hypothesis"], truncation=True, max_length=256
    ),
    batched=True,
    remove_columns=[
        column
        for column in test_dataset.column_names
        if column != "entailment_judgment"
    ],
).rename_column("entailment_judgment", "labels")
tokenized_test_dataset.set_format("torch")

test_loader = DataLoader(
    tokenized_test_dataset,
    batch_size=32,
    collate_fn=DataCollatorWithPadding(tokenizer=tokenizer),
)
predictions = []
probabilities = []
labels = []

with torch.no_grad():
    for batch in test_loader:
        batch_labels = batch.pop("labels")
        logits = model(
            **{name: values.to(device) for name, values in batch.items()}
        ).logits
        predictions.extend(logits.argmax(dim=-1).cpu().tolist())
        probabilities.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())
        labels.extend(batch_labels.tolist())

print(f"Test examples: {len(labels)}")
print(f"Accuracy: {accuracy_score(labels, predictions):.4f}")
print(f"ROC AUC: {roc_auc_score(labels, probabilities):.4f}")

Evaluating checkpoint: checkpoint-21951


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Test examples: 2448
Accuracy: 0.8317
ROC AUC: 0.9015


In [8]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

error_indices = np.flatnonzero(np.asarray(predictions) != np.asarray(labels))
incorrect_predictions = test_dataset.select(error_indices.tolist()).to_pandas()
incorrect_predictions = incorrect_predictions.assign(
    true_label=np.asarray(labels)[error_indices],
    predicted_label=np.asarray(predictions)[error_indices],
    predicted_entailment_probability=np.asarray(probabilities)[error_indices],
)

print(f"Incorrect predictions: {len(incorrect_predictions)} of {len(test_dataset)}")
display(
    incorrect_predictions[
        [
            "sentence_pair_id",
            "premise",
            "hypothesis",
            "true_label",
            "predicted_label",
            "predicted_entailment_probability",
        ]
    ]
)

Incorrect predictions: 412 of 2448


,sentence_pair_id,premise,hypothesis,true_label,predicted_label,predicted_entailment_probability
0,3,Um menino jovem vestindo um traje de banho vermelho está pulando em uma piscina de crianças azul,Um menino jovem está vestindo um maiô vermelho e pulando para fora de uma piscina de criança,0,1,0.999592
1,11,O monte de homens está brincando com lama em um campo de rúgbi,Alguns homens estão jogando rúgbi,0,1,0.994585
2,19,Dois meninos gêmeos pré-adolescentes estão duelando com varas,Dois meninos gêmeos pré-adolescentes estão brincando com cartas,0,1,0.999914
3,36,Um homem e uma mulher estão dirigindo pela estrada abaixo em um veículo sem teto,Dois homens estão dirigindo por uma estrada em um veículo conversível,0,1,0.999827
4,39,O homem está rachando ovos em uma tigela,A senhora está quebrando um ovo em uma tigela,0,1,0.999921
...,...,...,...,...,...,...
407,2431,Uma mulher de vestido vermelho guardando um instrumento,Uma mulher de vestido vermelho está tocando um instrumento,0,1,0.997846
408,2435,Uma menina de azul está descendo um escorregador verde,A menina jovem de azul está se divertindo em um balanço,0,1,0.998963
409,2441,Um menino e uma menina em trajes de banho estão usando boias em seus braços,O menino e a menina estão brincando e usando boias de braço,0,1,0.999418
410,2444,Uma pessoa de topless com uma mochila está em frente a uma pilha de pedras e nuvens estão ao fundo,O homem está de pé em uma montanha rochosa e nuvens cinzas estão ao fundo,0,1,0.998355
